<a href="https://colab.research.google.com/github/franciscorafael-dotcom/DL_P2_TP2_G2-F1/blob/main/notebooks/treino_cnn_4modelos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Treino - ResNet50, ConvNeXt-Tiny, DenseNet121 (CNN) e ViT-B/16 (Vision Transformer) (P2 · Tema F)
# Fine-tuning de quatro modelos pré-treinados (ImageNet) no FieldVitis + classe "Outro", com o `splits.json` do repositório (3 seeds).
# Alterações de 21/11: EfficientNet-B0 substituído pelo ViT-B/16; 5.ª classe "Outro" (imagens que não são folhas de videira).

In [2]:
# 0. (Só no Google Colab) Preparar o ambiente. Localmente, esta célula não faz nada.
import os
from pathlib import Path

try:
    import google.colab  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    try:
        import keras_hub  # necessário para o ViT
    except ImportError:
        !pip install -q keras-hub

    REPO = Path("/content/DL_P2_TP2_G2-F1")
    if not REPO.exists():
        !git clone -q https://github.com/franciscorafael-dotcom/DL_P2_TP2_G2-F1.git {REPO}

    # Imagens: FieldVitis (4 pastas) + classe Outro, a partir dos zips no Drive
    FV = REPO / "data" / "fieldvitis"
    FV.mkdir(parents=True, exist_ok=True)
    !unzip -q -o /content/drive/MyDrive/fieldvitis.zip -d {FV}
    !unzip -q -o /content/drive/MyDrive/outro.zip -d {FV}

    # Resultados (modelos .keras e CSV) no Drive, para não se perderem se o Colab desligar.
    # runs/P2_runs passa a ser um atalho para essa pasta; o resto do notebook não muda.
    DRIVE_RUNS = Path("/content/drive/MyDrive/P2_runs_cnn_vit")
    DRIVE_RUNS.mkdir(parents=True, exist_ok=True)
    destino = REPO / "runs" / "P2_runs"
    destino.parent.mkdir(parents=True, exist_ok=True)
    if destino.exists() and not destino.is_symlink():
        destino.rename(destino.with_name("P2_runs_do_git"))
    if not destino.exists():
        os.symlink(DRIVE_RUNS, destino)

    os.chdir(REPO / "notebooks")   # a célula seguinte encontra a raiz do repositório a partir daqui
    print("Pastas de imagens:", sorted(p.name for p in FV.iterdir() if p.is_dir()))
    print("Resultados em:", DRIVE_RUNS)

Mounted at /content/drive
Pastas de imagens: ['Black_rot', 'Esca', 'Healthy', 'Leaf_blight', 'Outro']
Resultados em: /content/drive/MyDrive/P2_runs_cnn_vit


In [3]:

# --- Configuração dos dados locais e dos splits ---
import json
from pathlib import Path
import pandas as pd

# Encontrar automaticamente a raiz do repositório.
# Funciona se o notebook for executado na raiz ou dentro de notebooks/.
cwd = Path.cwd().resolve()

REPO_DIR = next(
    (
        p for p in [cwd, *cwd.parents]
        if (p / "data" / "fieldvitis").is_dir()
        and (p / "data" / "splits.json").is_file()
    ),
    None
)

if REPO_DIR is None:
    raise FileNotFoundError(
        "Não foi possível encontrar a raiz do repositório."
    )

# Caminhos locais
DATA_DIR = REPO_DIR / "data" / "fieldvitis"

QUALITY_DIR = REPO_DIR / "data" / "qualidade"
RAW_MANIFEST_FILE = QUALITY_DIR / "manifest_raw.csv"
REMOVIDAS_FILE = QUALITY_DIR / "removidas.csv"

HOLDOUT_FILE = REPO_DIR / "data" / "holdout.csv"
MANIFEST_FILE = REPO_DIR / "data" / "manifest_clean.csv"
SPLITS_FILE = REPO_DIR / "data" / "splits.json"

# Confirmar que todos os ficheiros necessários existem
required_paths = [
    DATA_DIR,
    RAW_MANIFEST_FILE,
    REMOVIDAS_FILE,
    HOLDOUT_FILE,
    MANIFEST_FILE,
    SPLITS_FILE
]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(f"Não encontrado: {path}")

# Carregar os manifestos
manifest_raw = pd.read_csv(RAW_MANIFEST_FILE)
removidas = pd.read_csv(REMOVIDAS_FILE)
manifest = pd.read_csv(MANIFEST_FILE)

# Carregar os splits
with SPLITS_FILE.open(encoding="utf-8") as f:
    sj = json.load(f)

CLASSES = sj["classes"]
SEEDS = [0, 1, 2]

# Identificar o Holdout através das duas fontes
holdout_json = set(sj["holdout"])
holdout_csv = set(
    pd.read_csv(HOLDOUT_FILE)["path"].astype(str)
)

assert holdout_json == holdout_csv, (
    "Os caminhos do Holdout em splits.json e holdout.csv não coincidem."
)

HOLDOUT_PATHS = holdout_json

# Definir o conjunto elegível: manifesto limpo menos Holdout
clean_paths = set(manifest["path"].astype(str))

assert HOLDOUT_PATHS <= clean_paths, (
    "Há caminhos do Holdout que não constam do manifesto limpo."
)

pool_paths = clean_paths - HOLDOUT_PATHS

# Informação geral
print("Raiz do projeto:", REPO_DIR)
print("Dataset:", DATA_DIR)
print("Manifesto bruto:", len(manifest_raw))
print("Imagens removidas:", len(removidas))
print("Imagens no manifesto limpo:", len(clean_paths))
print("Imagens reservadas (Holdout):", len(HOLDOUT_PATHS))
print("Imagens elegíveis (sem Holdout):", len(pool_paths))

# Verificar os três conjuntos de cada seed
for seed in SEEDS:
    sp = sj["seeds"][str(seed)]

    split_sets = {
        k: set(sp[k])
        for k in ("train", "val", "test")
    }

    assert not (split_sets["train"] & split_sets["val"])
    assert not (split_sets["train"] & split_sets["test"])
    assert not (split_sets["val"] & split_sets["test"])

    assigned = set.union(*split_sets.values())

    assert assigned == pool_paths, (
        f"A seed {seed} não corresponde ao conjunto não-Holdout."
    )

    assert not (assigned & HOLDOUT_PATHS), (
        f"Fuga do Holdout detetada na seed {seed}!"
    )

    faltam = [
        p for p in assigned
        if not (DATA_DIR / p).is_file()
    ]

    print(
        f"Seed {seed}:",
        {k: len(v) for k, v in split_sets.items()},
        "| ficheiros em falta:", len(faltam),
        "| Holdout incluído: 0"
    )

    assert not faltam, (
        f"Há imagens dos splits que não existem em {DATA_DIR}."
    )

print("\nVerificação concluída: apenas imagens não-Holdout.")

Raiz do projeto: /content/DL_P2_TP2_G2-F1
Dataset: /content/DL_P2_TP2_G2-F1/data/fieldvitis
Manifesto bruto: 555
Imagens removidas: 0
Imagens no manifesto limpo: 555
Imagens reservadas (Holdout): 50
Imagens elegíveis (sem Holdout): 505
Seed 0: {'train': 352, 'val': 76, 'test': 77} | ficheiros em falta: 0 | Holdout incluído: 0
Seed 1: {'train': 352, 'val': 76, 'test': 77} | ficheiros em falta: 0 | Holdout incluído: 0
Seed 2: {'train': 352, 'val': 76, 'test': 77} | ficheiros em falta: 0 | Holdout incluído: 0

Verificação concluída: apenas imagens não-Holdout.


In [ ]:
# Proteção: confirmar que o splits.json inclui a classe Outro
assert "Outro" in CLASSES, "splits.json antigo (sem a classe Outro): enviar primeiro o novo para o GitHub"
print("Classes:", CLASSES)

In [4]:
# Configuração e Carregamento das imagens
import time, gc
import numpy as np, pandas as pd
import tensorflow as tf, keras
from keras import layers
from PIL import Image, ImageOps
from sklearn.metrics import f1_score, recall_score

IMG_SIZE, BATCH = 224, 32
EPOCHS_HEAD, LR_HEAD = 5, 1e-3     # fase 1: só a cabeça (base congelada)
EPOCHS_FT, LR_FT = 25, 1e-5        # fase 2: fine-tuning da rede toda
PATIENCE = 6                       # early stopping no Macro-F1 de validação
SEEDS = [0, 1, 2]
RUNS_DIR = REPO_DIR / "runs" / "P2_runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("Keras", keras.__version__, "| backend", keras.backend.backend(),
      "| GPU:", tf.config.list_physical_devices("GPU"))

# Todas as imagens carregadas uma vez: RGB, orientação EXIF corrigida, 224x224
all_paths = sorted({p for s in sj["seeds"].values() for k in ("train", "val", "test") for p in s[k]})
cls_idx = {c: i for i, c in enumerate(CLASSES)}
IMGS, LABELS = {}, {}
for p in all_paths:
    im = ImageOps.exif_transpose(Image.open(DATA_DIR / p)).convert("RGB")
    IMGS[p] = np.asarray(im.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR), dtype=np.uint8)
    LABELS[p] = cls_idx[Path(p).parts[0]]
print("Imagens carregadas:", len(IMGS), "| por classe:", np.bincount(list(LABELS.values())))

holdout_carregadas = len(set(IMGS) & HOLDOUT_PATHS)

print(f"Imagens Holdout carregadas: {holdout_carregadas}")

assert holdout_carregadas == 0, (
    "Foram carregadas imagens do Holdout!"
)

Keras 3.13.2 | backend tensorflow | GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Imagens carregadas: 505 | por classe: [ 35 182 144 144]
Imagens Holdout carregadas: 0


In [5]:
# Cada rede espera os píxeis num formato diferente (o mesmo usado no treino do ImageNet)
PREPROCESS = {
    "resnet50": keras.applications.resnet50.preprocess_input,           # converte para BGR e subtrai a média do ImageNet
    "efficientnetb0": keras.applications.efficientnet.preprocess_input,  # não altera: o modelo normaliza por dentro
    "convnext_tiny": keras.applications.convnext.preprocess_input,       # não altera: o modelo normaliza por dentro
    "densenet121": keras.applications.densenet.preprocess_input,         # escala para [0,1] e normaliza com média/desvio do ImageNet
}

def preparar_dados(seed, nome):
    """Devolve treino/val/teste da seed indicada, já pré-processados para o modelo."""
    sp = sj["seeds"][str(seed)]
    dados = {}
    for split in ["train", "val", "test"]:
        x = np.stack([IMGS[p] for p in sp[split]]).astype("float32")
        y = np.array([LABELS[p] for p in sp[split]])
        dados[split] = (PREPROCESS[nome](x), y, sp[split])
    return dados

In [6]:
def criar_modelo(nome):
    """Base pré-treinada no ImageNet + augmentation + camada final de 4 classes."""
    if nome == "resnet50":
        base = keras.applications.ResNet50(include_top=False, weights="imagenet",
                                           input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg")
    elif nome == "efficientnetb0":
        base = keras.applications.EfficientNetB0(include_top=False, weights="imagenet",
                                                 input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg")
    elif nome == "convnext_tiny":
        base = keras.applications.ConvNeXtTiny(include_top=False, weights="imagenet",
                                               input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg")
    else:
        base = keras.applications.DenseNet121(include_top=False, weights="imagenet",
                                              input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg")
    base.trainable = False   # começa congelada

    augmentation = keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.15),
        layers.RandomZoom(0.2),
    ])  # só atua durante o treino

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = augmentation(inputs)
    x = base(x, training=False)   # BatchNorm fica em modo inferência, mesmo no fine-tuning
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(len(CLASSES), activation="softmax")(x)
    return keras.Model(inputs, outputs), base

In [7]:
# ViT-B/16 (Vision Transformer, KerasHub): pré-processamento e construção do modelo
import keras_hub

VIT_PRESET = "vit_base_patch16_224_imagenet"   # ViT-B/16, pré-treinado no ImageNet-1k, entrada 224x224

# O conversor do preset aplica a mesma normalização usada no pré-treino do ViT
VIT_CONVERTER = keras_hub.layers.ViTImageConverter.from_preset(VIT_PRESET)

def preprocess_vit(x):
    return keras.ops.convert_to_numpy(VIT_CONVERTER(x))

PREPROCESS["vit_b16"] = preprocess_vit

def criar_modelo_vit():
    """Mesma estrutura das CNN: base pré-treinada + augmentation + camada final."""
    base = keras_hub.models.ViTBackbone.from_preset(VIT_PRESET)
    base.trainable = False   # começa congelada

    augmentation = keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.15),
        layers.RandomZoom(0.2),
    ])

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = augmentation(inputs)
    x = base(x, training=False)              # (batch, 197, 768): um vetor por patch de 16x16 + token de classe
    x = layers.GlobalAveragePooling1D()(x)   # média dos tokens -> um vetor de 768 por imagem
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(len(CLASSES), activation="softmax")(x)
    return keras.Model(inputs, outputs), base

# A função criar_modelo das CNN fica igual; esta só acrescenta o caso do ViT
if "_criar_modelo_cnn" not in globals():
    _criar_modelo_cnn = criar_modelo

def criar_modelo(nome):
    if nome == "vit_b16":
        return criar_modelo_vit()
    return _criar_modelo_cnn(nome)

In [8]:
# Treinar e Guardar
def guardar_previsoes(nome, seed, split, caminhos, y, probs):
    """CSV para o avaliador: uma linha por imagem, com a classe verdadeira, a prevista e as 4 probabilidades."""
    df = pd.DataFrame(probs, columns=[f"p_{c}" for c in CLASSES])
    df.insert(0, "y_pred", [CLASSES[i] for i in probs.argmax(axis=1)])
    df.insert(0, "y_true", [CLASSES[i] for i in y])
    df.insert(0, "path", caminhos)
    df.insert(0, "split", split)
    df.insert(0, "seed", seed)
    df.insert(0, "modelo", nome)
    df.to_csv(RUNS_DIR / f"preds_{nome}_seed{seed}_{split}.csv", index=False)

class EarlyStoppingMacroF1(keras.callbacks.Callback):
    """Early stopping no Macro-F1 de validação (maximizar); restaura os melhores pesos no fim."""
    def __init__(self, x_val, y_val, patience):
        super().__init__()
        self.x_val, self.y_val, self.patience = x_val, y_val, patience

    def on_train_begin(self, logs=None):
        self.best, self.wait, self.best_weights = -1.0, 0, None

    def on_epoch_end(self, epoch, logs=None):
        pred = self.model.predict(self.x_val, batch_size=BATCH, verbose=0).argmax(axis=1)
        f1 = f1_score(self.y_val, pred, average="macro")
        print(f"    val_macro_f1 = {f1:.4f} (melhor: {max(self.best, f1):.4f})")
        if f1 > self.best:
            self.best, self.wait = f1, 0
            self.best_weights = self.model.get_weights()
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.model.stop_training = True

    def on_train_end(self, logs=None):
        if self.best_weights is not None:
            self.model.set_weights(self.best_weights)

def treinar(nome, seed):
    keras.utils.set_random_seed(seed)   # reprodutibilidade
    dados = preparar_dados(seed, nome)
    x_tr, y_tr, _ = dados["train"]
    x_va, y_va, _ = dados["val"]

    # Pesos por classe: o black rot (poucas imagens) pesa mais na loss
    contagens = np.bincount(y_tr, minlength=len(CLASSES))
    class_weight = {i: len(y_tr) / (len(CLASSES) * n) for i, n in enumerate(contagens)}

    model, base = criar_modelo(nome)

    # Fase 1: treina só a camada final (base congelada)
    model.compile(optimizer=keras.optimizers.Adam(LR_HEAD),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(x_tr, y_tr, validation_data=(x_va, y_va), epochs=EPOCHS_HEAD,
              batch_size=BATCH, class_weight=class_weight, verbose=2)

    # Fase 2: fine-tuning de toda a rede, com learning rate baixo
    base.trainable = True
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    model.compile(optimizer=keras.optimizers.Adam(LR_FT),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    early = EarlyStoppingMacroF1(x_va, y_va, PATIENCE)
    model.fit(x_tr, y_tr, validation_data=(x_va, y_va), epochs=EPOCHS_FT,
              batch_size=BATCH, class_weight=class_weight, callbacks=[early], verbose=2)

    # Guardar o modelo (só no Drive) e as previsões de val e teste (para o avaliador)
    model.save(RUNS_DIR / f"{nome}_seed{seed}.keras")
    for split in ["val", "test"]:
        x, y, caminhos = dados[split]
        guardar_previsoes(nome, seed, split, caminhos, y, model.predict(x, verbose=0))

    _, val_acc = model.evaluate(x_va, y_va, verbose=0)
    print(f">>> {nome} seed {seed}: accuracy de validação = {val_acc:.3f}")

In [9]:
# ResNet50
for seed in SEEDS:
    print(f"\n===== ResNet50 | seed {seed} =====")
    treinar("resnet50", seed)
    keras.backend.clear_session()   # liberta a memória da GPU entre corridas


===== ResNet50 | seed 0 =====
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Epoch 1/5
11/11 - 19s - 2s/step - accuracy: 0.3750 - loss: 1.5144 - val_accuracy: 0.6974 - val_loss: 0.8866
Epoch 2/5
11/11 - 2s - 159ms/step - accuracy: 0.6364 - loss: 0.9629 - val_accuracy: 0.8684 - val_loss: 0.5000
Epoch 3/5
11/11 - 2s - 163ms/step - accuracy: 0.7812 - loss: 0.6001 - val_accuracy: 0.9079 - val_loss: 0.3414
Epoch 4/5
11/11 - 2s - 195ms/step - accuracy: 0.8239 - loss: 0.5081 - val_accuracy: 0.9342 - val_loss: 0.2611
Epoch 5/5
11/11 - 2s - 165ms/step - accuracy: 0.8665 - loss: 0.3764 - val_accuracy: 0.9474 - val_loss: 0.2185
Epoch 1/25
    val_macro_f1 = 0.9013 (melhor: 0.9013)
11/11 - 33s - 3s/step - accuracy: 0.8864 - loss: 0.3731 - val_accuracy: 0.9342 - val_loss: 0.2313
Epoch 2/25
    val_macro_f1 = 0.9563 (melhor: 0.9563)
11/11 - 5s - 413ms/step - accuracy: 0.9176 - loss: 0.2638 - val_accuracy: 0.9737 - val_loss: 0.1344
Epoch 3/25
    val_macro_f1 = 0.9160 (melhor: 0.9563)
11/11 - 5s

### EfficientNet-B0 retirado, ViT-B/16 no lugar

O EfficientNet-B0 foi retirado por ter o pior Macro-F1 de **validação** na versão com 4 classes (decisão de 21/11, registada no AI_LOG). No lugar entra um Vision Transformer (ViT-B/16), para comparar CNN com Transformer.

O ViT treina com **batch 16** (em vez de 32) por causa da memória da GPU; é a única diferença de hiperparâmetros em relação às CNN.

In [10]:
# ViT-B/16
BATCH_CNN = BATCH
BATCH = 16                      # ViT-B/16 tem ~86 M de parâmetros: batch menor para caber na memória da GPU
for seed in SEEDS:
    print(f"\n===== ViT-B/16 | seed {seed} =====")
    treinar("vit_b16", seed)
    keras.backend.clear_session()
BATCH = BATCH_CNN               # repõe o batch das CNN


===== ViT-B/16 | seed 0 =====
Epoch 1/5
22/22 - 19s - 868ms/step - accuracy: 0.4659 - loss: 1.4885 - val_accuracy: 0.7105 - val_loss: 0.7812
Epoch 2/5
22/22 - 7s - 308ms/step - accuracy: 0.6676 - loss: 0.8938 - val_accuracy: 0.8553 - val_loss: 0.5284
Epoch 3/5
22/22 - 7s - 312ms/step - accuracy: 0.7528 - loss: 0.6582 - val_accuracy: 0.8816 - val_loss: 0.4188
Epoch 4/5
22/22 - 7s - 315ms/step - accuracy: 0.7898 - loss: 0.6091 - val_accuracy: 0.8816 - val_loss: 0.3889
Epoch 5/5
22/22 - 7s - 315ms/step - accuracy: 0.8693 - loss: 0.4535 - val_accuracy: 0.9211 - val_loss: 0.2857
Epoch 1/25
    val_macro_f1 = 0.9264 (melhor: 0.9264)
22/22 - 58s - 3s/step - accuracy: 0.8693 - loss: 0.4166 - val_accuracy: 0.9474 - val_loss: 0.2007
Epoch 2/25
    val_macro_f1 = 0.9294 (melhor: 0.9294)
22/22 - 21s - 944ms/step - accuracy: 0.9432 - loss: 0.2511 - val_accuracy: 0.9605 - val_loss: 0.1444
Epoch 3/25
    val_macro_f1 = 0.9563 (melhor: 0.9563)
22/22 - 40s - 2s/step - accuracy: 0.9460 - loss: 0.2005 -

In [11]:
# ConvNeXt-Tiny
for seed in SEEDS:
    print(f"\n===== ConvNeXt-Tiny | seed {seed} =====")
    treinar("convnext_tiny", seed)
    keras.backend.clear_session()



===== ConvNeXt-Tiny | seed 0 =====
111650432/111650432 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
Epoch 1/5
11/11 - 27s - 2s/step - accuracy: 0.3466 - loss: 1.8270 - val_accuracy: 0.4868 - val_loss: 1.1369
Epoch 2/5
11/11 - 4s - 392ms/step - accuracy: 0.5483 - loss: 1.2673 - val_accuracy: 0.5921 - val_loss: 0.9183
Epoch 3/5
11/11 - 4s - 389ms/step - accuracy: 0.6080 - loss: 1.0894 - val_accuracy: 0.6316 - val_loss: 0.7894
Epoch 4/5
11/11 - 4s - 392ms/step - accuracy: 0.6420 - loss: 0.8932 - val_accuracy: 0.7895 - val_loss: 0.6165
Epoch 5/5
11/11 - 4s - 397ms/step - accuracy: 0.7727 - loss: 0.6323 - val_accuracy: 0.8684 - val_loss: 0.4861
Epoch 1/25
    val_macro_f1 = 0.7942 (melhor: 0.7942)
11/11 - 71s - 6s/step - accuracy: 0.7670 - loss: 0.7159 - val_accuracy: 0.8553 - val_loss: 0.4444
Epoch 2/25
    val_macro_f1 = 0.8368 (melhor: 0.8368)
11/11 - 16s - 1s/step - accuracy: 0.7869 - loss: 0.5530 - val_accuracy: 0.8947 - val_loss: 0.3917
Epoch 3/25
    val_macro_f1 = 0.8470 (melhor: 0.8470)
11/11

In [12]:
# DenseNet121
for seed in SEEDS:
    print(f"\n===== DenseNet121 | seed {seed} =====")
    treinar("densenet121", seed)
    keras.backend.clear_session()



===== DenseNet121 | seed 0 =====
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/5
11/11 - 26s - 2s/step - accuracy: 0.3153 - loss: 1.5762 - val_accuracy: 0.5395 - val_loss: 1.1809
Epoch 2/5
11/11 - 2s - 177ms/step - accuracy: 0.4631 - loss: 1.3131 - val_accuracy: 0.7105 - val_loss: 0.9814
Epoch 3/5
11/11 - 2s - 184ms/step - accuracy: 0.5909 - loss: 1.0402 - val_accuracy: 0.8026 - val_loss: 0.8017
Epoch 4/5
11/11 - 2s - 182ms/step - accuracy: 0.6477 - loss: 0.9386 - val_accuracy: 0.8289 - val_loss: 0.6953
Epoch 5/5
11/11 - 2s - 173ms/step - accuracy: 0.6989 - loss: 0.7987 - val_accuracy: 0.8421 - val_loss: 0.6264
Epoch 1/25
    val_macro_f1 = 0.7618 (melhor: 0.7618)
11/11 - 49s - 4s/step - accuracy: 0.7784 - loss: 0.7517 - val_accuracy: 0.8421 - val_loss: 0.4916
Epoch 2/25
    val_macro_f1 = 0.8699 (melhor: 0.8699)
11/11 - 5s - 500ms/step - accuracy: 0.7727 - loss: 0.6454 - val_accuracy: 0.9211 - val_loss: 0.3228
Epoch 3/25
    val_macro_f1 = 0.9060 (melhor: 0.9060)
11/11 -

In [13]:
import glob
for f in sorted(glob.glob(str(RUNS_DIR / "preds_*_val.csv"))):
    df = pd.read_csv(f)
    print(f"{f.split('/')[-1]:40s} accuracy val = {(df.y_true == df.y_pred).mean():.3f}")

preds_convnext_tiny_seed0_val.csv        accuracy val = 0.947
preds_convnext_tiny_seed1_val.csv        accuracy val = 0.934
preds_convnext_tiny_seed2_val.csv        accuracy val = 0.908
preds_densenet121_seed0_val.csv          accuracy val = 0.974
preds_densenet121_seed1_val.csv          accuracy val = 0.974
preds_densenet121_seed2_val.csv          accuracy val = 0.974
preds_resnet50_seed0_val.csv             accuracy val = 0.974
preds_resnet50_seed1_val.csv             accuracy val = 0.987
preds_resnet50_seed2_val.csv             accuracy val = 0.987
preds_vit_b16_seed0_val.csv              accuracy val = 0.974
preds_vit_b16_seed1_val.csv              accuracy val = 1.000
preds_vit_b16_seed2_val.csv              accuracy val = 0.974


In [14]:
# Métricas de validação: Macro-F1 nas 4 doenças + desempenho da classe "Outro" (média ± desvio padrão das 3 seeds)
DOENCAS = [c for c in CLASSES if c != "Outro"]

def metricas(split):
    linhas = []
    for f in sorted(RUNS_DIR.glob(f"preds_*_{split}.csv")):
        df = pd.read_csv(f)
        e_outro = df.y_true == "Outro"
        folhas = df[~e_outro]
        linhas.append({
            "modelo": df.modelo[0], "seed": int(df.seed[0]),
            # Métrica principal (comparável com o VLM): Macro-F1 só nas 4 doenças.
            # Uma imagem "Outro" classificada como doença conta como falso positivo dessa doença.
            "macro_f1_doencas": f1_score(df.y_true, df.y_pred, labels=DOENCAS, average="macro"),
            "accuracy_folhas": (folhas.y_true == folhas.y_pred).mean(),
            "outro_rejeitadas": (df.y_pred[e_outro] == "Outro").mean(),   # não-folhas bem rejeitadas
            "folhas_rejeitadas": (folhas.y_pred == "Outro").mean(),      # folhas rejeitadas por engano
        })
    return pd.DataFrame(linhas)

met_val = metricas("val")
display(met_val.round(4))
display(met_val.groupby("modelo").agg(["mean", "std"]).drop(columns="seed").round(4))

,modelo,seed,macro_f1_doencas,accuracy_folhas,outro_rejeitadas,folhas_rejeitadas
0,convnext_tiny,0,0.9172,0.9474,NaN,0.0
1,convnext_tiny,1,0.8932,0.9342,NaN,0.0
2,convnext_tiny,2,0.8809,0.9079,NaN,0.0
3,densenet121,0,0.9563,0.9737,NaN,0.0
4,densenet121,1,0.9467,0.9737,NaN,0.0
5,densenet121,2,0.9608,0.9737,NaN,0.0
6,resnet50,0,0.9563,0.9737,NaN,0.0
7,resnet50,1,0.9667,0.9868,NaN,0.0
8,resnet50,2,0.9715,0.9868,NaN,0.0
9,vit_b16,0,0.9563,0.9737,NaN,0.0


macro_f1_doencas         accuracy_folhas          \
                          mean     std            mean     std   
modelo                                                           
convnext_tiny           0.8971  0.0185          0.9298  0.0201   
densenet121             0.9546  0.0072          0.9737  0.0000   
resnet50                0.9648  0.0077          0.9825  0.0076   
vit_b16                 0.9724  0.0240          0.9825  0.0152   

              outro_rejeitadas     folhas_rejeitadas       
                          mean std              mean  std  
modelo                                                     
convnext_tiny              NaN NaN               0.0  0.0  
densenet121                NaN NaN               0.0  0.0  
resnet50                   NaN NaN               0.0  0.0  
vit_b16                    NaN NaN               0.0  0.0

### Atenção: conjunto de teste

As células seguintes leem as previsões de **teste**. Só devem ser corridas depois de o `PROTOCOLO.md` estar atualizado (5 classes, ViT, holdout) e com commit, e nenhuma decisão sobre os modelos pode ser tomada a partir delas. Para as métricas completas no teste: `metricas("test")`.

In [15]:
# Métricas por seed e média das 3 seeds (conjunto de teste)
from sklearn.metrics import accuracy_score
linhas = []
for f in sorted(glob.glob(str(RUNS_DIR / "preds_*_test.csv"))):
    df = pd.read_csv(f)
    linhas.append({"modelo": df.modelo[0], "seed": df.seed[0],
                   "accuracy": accuracy_score(df.y_true, df.y_pred),
                   "macro_f1": f1_score(df.y_true, df.y_pred, average="macro"),
                   "macro_recall": recall_score(df.y_true, df.y_pred, average="macro")})
por_seed = pd.DataFrame(linhas)
print("Métricas por seed:")
print(por_seed.round(4).to_string(index=False))
print("\nMédia das 3 seeds por modelo:")
print(por_seed.groupby("modelo")[["accuracy", "macro_f1", "macro_recall"]].mean().round(4))


Métricas por seed:
       modelo  seed  accuracy  macro_f1  macro_recall
convnext_tiny     0    0.9091    0.8733        0.8902
convnext_tiny     1    0.8831    0.8400        0.8998
convnext_tiny     2    0.9481    0.9125        0.9545
  densenet121     0    0.9610    0.9334        0.9680
  densenet121     1    0.9481    0.9125        0.9545
  densenet121     2    0.9481    0.9443        0.9609
     resnet50     0    0.9870    0.9717        0.9583
     resnet50     1    0.9740    0.9643        0.9773
     resnet50     2    0.9481    0.9137        0.9545
      vit_b16     0    0.9610    0.9543        0.9680
      vit_b16     1    0.9221    0.8728        0.8733
      vit_b16     2    0.9610    0.9421        0.9680

Média das 3 seeds por modelo:
               accuracy  macro_f1  macro_recall
modelo                                         
convnext_tiny    0.9134    0.8753        0.9148
densenet121      0.9524    0.9301        0.9611
resnet50         0.9697    0.9499        0.9634
vit_b16 

In [16]:
# Custo computacional: parâmetros e latência de inferência (modelos da seed 0, imagens de validação)
custo = []
for nome in ["resnet50", "convnext_tiny", "densenet121", "vit_b16"]:
    model = keras.models.load_model(RUNS_DIR / f"{nome}_seed0.keras")
    x, _, _ = preparar_dados(0, nome)["val"]

    model.predict(x[:BATCH], verbose=0)                 # aquecimento
    t0 = time.perf_counter()
    model.predict(x, batch_size=BATCH, verbose=0)
    ms_lote = (time.perf_counter() - t0) / len(x) * 1000

    model(x[:1], training=False)                        # aquecimento
    t0 = time.perf_counter()
    for i in range(20):
        model(x[i:i+1], training=False)
    ms_uma = (time.perf_counter() - t0) / 20 * 1000

    custo.append(dict(modelo=nome,
                      parametros_milhoes=round(model.count_params() / 1e6, 2),
                      ms_por_imagem_em_lote=round(ms_lote, 1),
                      ms_por_imagem_isolada=round(ms_uma, 1)))
    del model; keras.backend.clear_session(); gc.collect()

display(pd.DataFrame(custo))

,modelo,parametros_milhoes,ms_por_imagem_em_lote,ms_por_imagem_isolada
0,resnet50,23.60,29.7,258.5
1,convnext_tiny,27.82,67.6,2894.2
2,densenet121,7.04,53.1,505.0
3,vit_b16,85.80,40.3,195.3


In [17]:
# Limiar de confiança para a app (escolhido na VALIDAÇÃO do modelo usado no predict.py)
# A classe "Outro" não cobre todas as imagens possíveis; o limiar apanha o que escapar.
MODELO_APP, SEED_APP = "resnet50", 1    # tem de coincidir com o MODEL_PATH do predict.py

df = pd.read_csv(RUNS_DIR / f"preds_{MODELO_APP}_seed{SEED_APP}_val.csv")
conf = df[[f"p_{c}" for c in CLASSES]].max(axis=1)
folha_certa = (df.y_true != "Outro") & (df.y_true == df.y_pred)

# Limiar = valor abaixo do qual ficam só 5% das folhas bem classificadas (aceita 95% delas)
LIMIAR = round(float(np.quantile(conf[folha_certa], 0.05)), 2)

aceites = (conf >= LIMIAR) & (df.y_pred != "Outro")
print(f"Limiar escolhido: {LIMIAR}  -> copiar para LIMIAR_CONFIANCA no predict.py")
print(f"Folhas aceites (validação): {aceites[df.y_true != 'Outro'].mean():.1%}")
print(f"Imagens 'Outro' rejeitadas (validação): {(~aceites[df.y_true == 'Outro']).mean():.1%}")

Limiar escolhido: 0.98  -> copiar para LIMIAR_CONFIANCA no predict.py
Folhas aceites (validação): 93.4%
Imagens 'Outro' rejeitadas (validação): nan%
